# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.2 of 7 — Submit Batch Jobs and Monitor Status

**Role in the pipeline:**
This notebook has two responsibilities run in sequence:
1. **Submit** JSONL batch input files to the OpenAI Batch API to create classification jobs.
2. **Poll and download** — once jobs complete, download the output JSONL files.

**⚠️ Asynchronous step — must be run twice:**
- **First run:** executes section 3.2.3 to submit jobs. All new `input_batch_status == 'created'` rows without a job ID are submitted. The notebook may then be closed.
- **Second run (after Batch API completes, typically within 24 hours):** re-run section 3.2.4 to poll job status and download completed output files.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. ▶ **3.2 — Submit batch jobs and monitor status** ← *you are here*
3. 3.3 — Extract classification results from batch output
4. 3.4 — Split missing and complete classifications
5. 3.5 — Create batch input files for missing records
6. 3.6 — Submit batch jobs for missing records *(run after Batch API completes)*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Notebook 3.1 completed — JSONL input files exist in `../data/stage_03/intermediate/input/`
- `OPENAI_API_KEY` set in `../.env`

## 3.2.1. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [1]:
import pandas as pd
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules, the OpenAI client, and JSON utilities needed for reading model names from JSONL files.

In [2]:
import stage2 as st2
import stage3 as st3
from openai import OpenAI
import json
import pandas as pd

## 3.2.2. Load processing logs

Reads the current Stage 3 tracker from disk. The commented-out lines show how to reset specific status columns if re-processing is needed (e.g., resetting `missing_job_status` before a retry run).

In [3]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df

,input_file,extract_path,input_batch_path,input_batch_status,job_id,job_status,output_batch_path,output_batch_status,result_path,result_status
0,ua-2024-01-01,../data/stage_02/processed/output\ua-2024-01-0...,../data/stage_03/raw/input\ua-2024-01-01.jsonl,created,NaN,NaN,NaN,NaN,NaN,NaN


## 3.2.3. Create Batch Jobs

Initialises the OpenAI client using the API key from `.env`.

In [4]:
client = OpenAI(api_key=g.Config.OPENAI_API_KEY)

**Alert:** `_from` and `_to` control which rows are submitted. For a fresh full run set `_to = process_df.shape[0]`. The OpenAI Batch API has a limit on concurrent jobs — submit in batches if needed.

In [5]:
_from = 0
_to = 1

Job submission loop: iterates over rows in `[_from:_to]`. Rows already `completed`, `in_progress`, `finalizing`, or `validating` are skipped. For eligible rows, `create_batch_job()` uploads the JSONL file and creates a Batch API job. The returned job ID and status `in_progress` are written back to the tracker immediately.

In [6]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)

for _,row in process_df[_from:_to].iterrows():
    if row["input_batch_status"] == "created":
        if row["job_status"] == "completed" or row["job_status"] == "in_progress" or row["job_status"] == "finalizing" or row["job_status"] == "validating":
            continue
        else:
            batch_job = st3.create_batch_job(client, row["input_batch_path"])
            process_df.loc[_, "job_id"] = batch_job.id
            process_df.loc[_, "job_status"] = "in_progress"
            process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)
            print("Added job for: ", str(_), "-", row["input_file"])

Added job for:  0 - ua-2024-01-01


## 3.2.4. Check Jobs and Download Files

A local utility that reads the model name from the first line of a batch JSONL file — used only for diagnostic print output in the polling loop below.

In [7]:
def get_model_from_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        first_line = f.readline()
    data = json.loads(first_line)
    return data["body"]["model"]

**Polling and download loop** — run this after the Batch API reports jobs as complete (typically within 24 hours). For each row with `job_status == 'in_progress'` (or `finalizing`/`validating`):

1. Retrieves the current batch status from the API.
2. If `completed`: downloads the output JSONL content, writes it to `STAGE3_OUTPUT_PATH`, and updates `output_batch_status = 'complete'`.
3. If still running: prints progress (rows completed vs. total).
4. If error: prints the status and row index for investigation.

The tracker is saved after every row update to allow safe interruption and resumption.

In [9]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)

index = 0

for _,row in process_df[_from:_to].iterrows():

    if row["job_status"] == "in_progress" or row["job_status"] == "finalizing" or row["job_status"] == "validating":
        index += 1
        try:
            batch = client.batches.retrieve(row["job_id"])
            print("row:", _, "-",batch.status, "-", row["job_id"])
        except ValueError as e:
            process_df.loc[_, "job_status"] = None
            process_df.loc[_, "job_id"] = None

        process_df.loc[_, "job_status"] = batch.status

        if batch.status == "completed":
            result_file_id = batch.output_file_id
            result = client.files.content(result_file_id).content
            output_path = os.path.join(g.Config.STAGE3_OUTPUT_PATH, process_df.loc[_, "input_file"]  + ".json")
            with open(output_path, 'wb') as file:
                file.write(result)
            process_df.loc[_, "output_batch_path"] = output_path
            process_df.loc[_, "output_batch_status"] = "complete"
            print(index, "Added completed batch file for: ", row["input_file"])
        elif batch.status != "in_progress" and batch.status != "finalizing":
            print(index, "Error: ", row["input_file"], " Status: ", batch.status, "-", get_model_from_jsonl(row["input_batch_path"]), " - ", batch.request_counts.total,  " rows, ", f"row({_})")
        else:
            print(index, "Waiting for: ", row["input_file"], " Status: ", batch.status, "-", batch.request_counts.completed, "of", batch.request_counts.total, "-", get_model_from_jsonl(row["input_batch_path"]), f"row ({_})")

        process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)

row: 0 - completed - batch_69c055917b0481909b82ddb32c228250
1 Added completed batch file for:  ua-2024-01-01


---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.